# Workshop 2: Задачи по теми 3 и 4 (ООП и функционално програмиране)

## Задача 1 – `Fraction`

Напишете клас `Fraction`, който представя обикновена дроб и се държи като "истинско" число в Python.

**Създаване и представяне**
* `Fraction(numerator, denominator=1)` – дробта винаги се пази **съкратена**, а знакът е в числителя (`Fraction(2, -4)` е същото като `Fraction(-1, 2)`). Приемаме, че знаменателят никога не е 0.
* `numerator` и `denominator` – само за четене
* клас-метод `Fraction.parse(text)`, който приема низове като `"3/4"`, `"-5"`, `" 2 / -6 "`
* `repr` → `"Fraction(1, 2)"`, `str` → `"1/2"` (или само `"3"`, ако знаменателят е 1)

**Аритметика**
* `+`, `-`, `*`, `/` между две дроби, **както и между дроб и `int` – от двете страни** (`1 + Fraction(1, 2)`, `Fraction(1, 2) * 3`)
* унарен минус, `abs(...)`, `**` с цяла (и отрицателна!) степен
* `sum([...])` върху списък от дроби трябва да работи

**Сравнение и конвертиране**
* `==`, `!=`, `<`, `<=`, `>`, `>=` – с дроби и с `int`
* `Fraction(4, 2) == 2` и `hash(Fraction(4, 2)) == hash(2)` (равни обекти трябва да имат равни хешове)
* сравнение с други типове (напр. `str`) не трябва да гърми, а просто да е `False` за `==`
* `float(...)`, `int(...)` (отрязва дробната част, като `int(-3.5)`), `bool(...)`

* Hint: `math.gcd`, `__radd__` & co., `NotImplemented`

In [ ]:
class Fraction:
    pass  # your code here


# Tests:

half, third = Fraction(1, 2), Fraction(1, 3)

# creation
assert Fraction(2, 4) == half
assert (Fraction(2, -4).numerator, Fraction(2, -4).denominator) == (-1, 2), "the sign goes in the numerator"
assert isinstance(Fraction.numerator, property) and Fraction.numerator.fset is None, "numerator should be read-only"
assert Fraction.parse("3/4") == Fraction(3, 4)
assert Fraction.parse("-5") == Fraction(-5)
assert Fraction.parse(" 2 / -6 ") == Fraction(-1, 3)
assert repr(half) == "Fraction(1, 2)" and str(half) == "1/2" and str(Fraction(6, 3)) == "2"

# arithmetic
assert half + third == Fraction(5, 6)
assert half - third == Fraction(1, 6)
assert half * third == Fraction(1, 6)
assert half / third == Fraction(3, 2)
assert half + 1 == Fraction(3, 2) and 1 + half == Fraction(3, 2), "int on both sides"
assert 1 - half == half and 2 * third == Fraction(2, 3) and 1 / third == 3
assert -half == Fraction(-1, 2) and abs(Fraction(-3, 4)) == Fraction(3, 4)
assert Fraction(2, 3) ** 2 == Fraction(4, 9)
assert Fraction(2, 3) ** -2 == Fraction(9, 4)
assert Fraction(-2, 3) ** -1 == Fraction(-3, 2), "the sign must stay in the numerator"
assert sum([half, half, third, third, third]) == 2

# comparison
assert third < half <= half < 1 and 1 > half >= third
assert Fraction(4, 2) == 2 and 2 == Fraction(4, 2)
assert half != third and half != 1
assert half != "1/2", "comparison with other types should not crash"
assert sorted([Fraction(3, 4), 0, third, half]) == [0, third, half, Fraction(3, 4)]
assert len({Fraction(1, 2), Fraction(2, 4), Fraction(-3, -6)}) == 1
assert hash(Fraction(4, 2)) == hash(2)
assert len({2, Fraction(4, 2)}) == 1

# conversion
assert float(Fraction(3, 4)) == 0.75
assert int(Fraction(7, 2)) == 3 and int(Fraction(-7, 2)) == -3
assert not Fraction(0, 5) and Fraction(1, 5)

"✅ All OK!"

## Задача 2 – Библиотека

Моделирайте система за библиотека. Времето ще меряме в **дни** (цели числа, ден 0, ден 1, ...).

**Абстрактен клас `Item(title)`**
* всеки създаден обект (от който и да е наследник) получава уникален `id` – поредно число, с 1 по-голямо от това на предишния създаден обект
* абстрактни property-та `loan_days` (за колко дни се заема) и `daily_fee` (глоба на ден закъснение)
* клас-атрибут `MAX_FEE` – максимална глоба за един заем (`None` = без таван)
* метод `late_fee(days_late)` – глобата за даден брой дни закъснение, съобразена с `MAX_FEE`
* `str(item)` → `"[#<id>] <title> (<тип>)"`, напр. `"[#3] Dune (Book)"` – **без** да се предефинира в наследниците

**Наследници**

| Клас | `loan_days` | `daily_fee` | `MAX_FEE` |
|---|---|---|---|
| `Book(title, author)` | 21 | 0.25 | 10 |
| `DVD(title, minutes)` | 7 | 1.0 | 20 |
| `Magazine(title, issue)` | 3 | 0.5 | без таван |

**Клас `Library`**
* `add(item)`, `len(library)`, `item in library`, `for item in library` (всички обекти, в реда на добавяне)
* `available()` – **генератор** на обектите, които не са заети в момента
* `checkout(item_id, member, day)` – заема обекта. Връща `True` при успех и `False`, ако: няма такъв обект; той вече е зает; членът вече има 3 заети обекта; или членът дължи повече от 5 лв.
* `return_item(item_id, day)` – връща обекта и **връща глобата** (0, ако няма закъснение). Глобата се добавя към дълга на члена. Ако обектът не е бил зает – връща `None`.
* `pay(member, amount)` – намалява дълга на члена (не под 0)
* `loans_of(member)` – сортиран списък от заглавията, заети от члена
* `debtors()` – речник `член → дълг` само за членовете с дълг > 0

In [ ]:
from abc import ABC, abstractmethod

# your code here


# Tests:

dune = Book("Dune", "Frank Herbert")
matrix = DVD("The Matrix", 136)
ng = Magazine("National Geographic", 42)
hobbit = Book("The Hobbit", "J. R. R. Tolkien")
alien = DVD("Alien", 117)

assert matrix.id == dune.id + 1 and ng.id == dune.id + 2 and alien.id == dune.id + 4, "ids are shared by all item types"
assert str(dune) == f"[#{dune.id}] Dune (Book)"
assert str(ng) == f"[#{ng.id}] National Geographic (Magazine)"
assert "__str__" not in vars(Book), "implement __str__ only in Item"
assert Item.__abstractmethods__ == frozenset({"loan_days", "daily_fee"})

assert dune.late_fee(0) == 0 and dune.late_fee(4) == 1.0 and dune.late_fee(1000) == 10
assert matrix.late_fee(3) == 3.0 and matrix.late_fee(50) == 20
assert ng.late_fee(1000) == 500.0, "magazines have no fee cap"

lib = Library()
for item in [dune, matrix, ng, hobbit]:
    lib.add(item)

assert len(lib) == 4 and dune in lib and alien not in lib
assert [i.title for i in lib] == ["Dune", "The Matrix", "National Geographic", "The Hobbit"]

assert lib.checkout(dune.id, "ana", day=0)
assert not lib.checkout(dune.id, "bob", day=0), "already loaned"
assert not lib.checkout(alien.id, "bob", day=0), "not in this library"
assert lib.checkout(matrix.id, "ana", day=1)
assert lib.checkout(ng.id, "ana", day=1)
assert not lib.checkout(hobbit.id, "ana", day=2), "ana already has 3 items"
assert lib.loans_of("ana") == ["Dune", "National Geographic", "The Matrix"]
assert lib.loans_of("bob") == []

available = lib.available()
assert next(available) is hobbit
assert [i.title for i in lib.available()] == ["The Hobbit"]

assert lib.return_item(dune.id, day=21) == 0, "returned on time"
assert lib.return_item(dune.id, day=22) is None, "not loaned anymore"
assert lib.return_item(ng.id, day=10) == 3.0  # due on day 4, 6 days late
assert lib.return_item(matrix.id, day=12) == 4.0  # due on day 8, 4 days late
assert lib.debtors() == {"ana": 7.0}

assert not lib.checkout(hobbit.id, "ana", day=12), "ana owes more than 5"
lib.pay("ana", 2)
assert lib.checkout(hobbit.id, "ana", day=12), "a debt of exactly 5 is fine"
lib.pay("ana", 100)
assert lib.debtors() == {}

"✅ All OK!"

## Задача 3 – `memoize` и `trace`

**`@memoize`**
* кешира резултатите на функцията по позиционните ѝ аргументи (приемаме, че са hashable)
* атрибути `hits` и `misses` на декорираната функция – колко пъти резултатът е взет от кеша и колко пъти е сметнат наново
* метод `cache_clear()`, който изчиства кеша и нулира броячите
* декорираната функция пази името си (`__name__`)

**`@trace`**
* записва всяко извикване в атрибута `log` (списък от низове) на декорираната функция:
  * при влизане: `"<име>(<аргументи>)"`, аргументите са с `repr` и разделени със `", "`, а keyword аргументите са като `key=value`
  * при излизане: `"-> <repr на резултата>"`
  * всеки ред е отместен с по 2 интервала за всяко ниво на вложеност (рекурсия)

Например за
```python
@trace
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

fib(3)
print("\n".join(fib.log))
```
```
fib(3)
  fib(2)
    fib(1)
    -> 1
    fib(0)
    -> 0
  -> 1
  fib(1)
  -> 1
-> 2
```

Помислете какво става, когато комбинирате двата декоратора – има ли значение редът им?

* Hint: `functools.wraps`

In [ ]:
def memoize(func):
    pass  # your code here


def trace(func):
    pass  # your code here


# Tests:

@trace
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

assert fib(3) == 2
assert fib.log == ["fib(3)", "  fib(2)", "    fib(1)", "    -> 1", "    fib(0)", "    -> 0", "  -> 1", "  fib(1)", "  -> 1", "-> 2"]
assert fib.__name__ == "fib"

@trace
def greet(name, punctuation="!"):
    return f"Hi, {name}{punctuation}"

greet("Ana", punctuation="?")
assert greet.log == ["greet('Ana', punctuation='?')", "-> 'Hi, Ana?'"]


@memoize
def slow_fib(n):
    return n if n < 2 else slow_fib(n - 1) + slow_fib(n - 2)

assert slow_fib(30) == 832040
assert (slow_fib.misses, slow_fib.hits) == (31, 28)
assert slow_fib(30) == 832040 and slow_fib.hits == 29
slow_fib.cache_clear()
assert (slow_fib.misses, slow_fib.hits) == (0, 0)
slow_fib(2)
assert (slow_fib.misses, slow_fib.hits) == (3, 0), "the cache should really be empty after cache_clear"
assert slow_fib.__name__ == "slow_fib"


# memoize outside: only the cache misses reach trace
@memoize
@trace
def fib_a(n):
    return n if n < 2 else fib_a(n - 1) + fib_a(n - 2)

# trace outside: every call is logged, even the cached ones
@trace
@memoize
def fib_b(n):
    return n if n < 2 else fib_b(n - 1) + fib_b(n - 2)

fib_a(10)
fib_b(10)
assert len(fib_a.log) == 2 * 11
assert len(fib_b.log) == 2 * 19
assert fib_a.log[-1] == fib_b.log[-1] == "-> 55"

"✅ All OK!"

## Задача 4 – Мързеливи заявки

Напишете клас `Query(iterable)`, с който да строим вериги от операции върху поредици:

```python
Query(range(100)).where(lambda x: x % 3 == 0).select(lambda x: x * x).skip(1).take(3).to_list()
# [9, 36, 81]
```

**Операции** – всяка връща **нов** `Query`:
* `where(predicate)` – само елементите, за които `predicate` е истина
* `select(func)` – прилага `func` върху всеки елемент
* `take(n)` / `skip(n)` – първите `n` елемента / всички без първите `n`
* `distinct(key=None)` – премахва повторенията (сравнява по `key(x)`, ако е даден), пазейки първата поява
* `chunk(size)` – групира елементите в списъци с дължина `size` (последният може да е по-къс)
* `flatten()` – "разгъва" едно ниво на вложеност: `[[1, 2], [3]]` → `1, 2, 3`

**Крайни операции:**
* `for x in query`
* `to_list()`, `count()`
* `first(default=None)` – първият елемент или `default`, ако няма такъв

⚠️ Всичко трябва да е **мързеливо** – нищо не се изчислява, докато не започнем да итерираме, и се изчислява само толкова, колкото е нужно. Това означава, че трябва да работи и с безкрайни поредици (`itertools.count()`).

In [ ]:
class Query:
    pass  # your code here


# Tests:

import itertools

assert Query(range(100)).where(lambda x: x % 3 == 0).select(lambda x: x * x).skip(1).take(3).to_list() == [9, 36, 81]
assert Query([3, 1, 3, 2, 1]).distinct().to_list() == [3, 1, 2]
assert Query(["apple", "Avocado", "banana", "Blueberry", "cherry"]).distinct(key=lambda s: s[0].lower()).to_list() == ["apple", "banana", "cherry"]
assert Query(range(7)).chunk(3).to_list() == [[0, 1, 2], [3, 4, 5], [6]]
assert Query([[1, 2], [], [3]]).flatten().to_list() == [1, 2, 3]
assert Query("hello").chunk(2).select("".join).to_list() == ["he", "ll", "o"]
assert Query(range(10)).where(lambda x: x > 100).first() is None
assert Query(range(10)).where(lambda x: x > 100).first(default=-1) == -1
assert Query(range(10)).skip(3).count() == 7
assert [x for x in Query("abc").select(str.upper)] == ["A", "B", "C"]

# each operation returns a new Query, the original stays the same
numbers = Query(range(5))
evens = numbers.where(lambda x: x % 2 == 0)
assert isinstance(evens, Query) and evens is not numbers
assert numbers.to_list() == [0, 1, 2, 3, 4]
assert evens.to_list() == [0, 2, 4]

# laziness
calls = []
def spy(x):
    calls.append(x)
    return x

q = Query(itertools.count()).select(spy).where(lambda x: x % 2).take(3)
assert calls == [], "nothing should be computed before iterating"
assert q.to_list() == [1, 3, 5]
assert calls == [0, 1, 2, 3, 4, 5], "only as much as needed should be computed"

assert Query(itertools.count(1)).select(lambda x: [x] * x).flatten().chunk(4).skip(2).first() == [4, 4, 5, 5]
assert Query(itertools.count()).select(lambda x: x % 5).distinct().take(5).to_list() == [0, 1, 2, 3, 4]

"✅ All OK!"

## Задача 5 – Символни изрази

Израз представяме като дърво:
* число (`int` или `float`)
* променлива – низ, напр. `"x"`
* `(op, a, b)`, където `op` е `"+"`, `"-"`, `"*"` или `"/"`, а `a` и `b` са изрази
* `("neg", a)` – отрицание на израза `a`

Например $ 2 \cdot x + (-y) $ е `("+", ("*", 2, "x"), ("neg", "y"))`.

Напишете следните функции, като използвате **`match`**:

**`evaluate(expr, env)`** – пресмята израза, като стойностите на променливите са в речника `env`. Връща `None`, ако изразът е невалиден, има непозната променлива или деление на 0.

**`to_string(expr)`** – връща израза като низ. Всяка бинарна операция е в скоби: `"((2 * x) + -y)"`. Отрицанието е `"-"` пред израза.

**`simplify(expr)`** – опростява израза рекурсивно (първо децата, после текущия възел) по следните правила:
* операция само върху числа се пресмята (с изключение на деление на 0 – тогава изразът остава какъвто е)
* `x + 0`, `0 + x`, `x - 0`, `x * 1`, `1 * x`, `x / 1` → `x`
* `x * 0`, `0 * x`, `0 / x` → `0`
* `x - x` → `0` и `x / x` → `1` (когато двете страни са еднакви изрази)
* `0 - x` → `("neg", x)`
* `("neg", ("neg", x))` → `x`

In [ ]:
def evaluate(expr, env):
    pass  # your code here


def to_string(expr):
    pass  # your code here


def simplify(expr):
    pass  # your code here


# Tests:

expr = ("+", ("*", 2, "x"), ("neg", "y"))
env = {"x": 5, "y": 3}

assert evaluate(42, {}) == 42
assert evaluate("x", env) == 5
assert evaluate(expr, env) == 7
assert evaluate(("/", "x", 2), env) == 2.5
assert evaluate(("-", ("neg", "x"), ("*", "y", "y")), env) == -14
assert evaluate("z", env) is None, "unknown variable"
assert evaluate(("/", 1, ("-", "x", 5)), env) is None, "division by zero"
assert evaluate(("%", 1, 2), env) is None, "unknown operation"
assert evaluate(("+", 1), env) is None, "wrong number of arguments"
assert evaluate(("+", 1, ("+", "x", "z")), env) is None, "errors should propagate"

assert to_string(expr) == "((2 * x) + -y)"
assert to_string(("neg", ("-", "a", 1.5))) == "-(a - 1.5)"
assert to_string("x") == "x" and to_string(7) == "7"

assert simplify(("+", 2, 3)) == 5
assert simplify(("*", ("+", 1, 1), ("-", 10, 7))) == 6
assert simplify(("+", "x", 0)) == "x"
assert simplify(("*", 1, ("+", "x", ("*", "y", 0)))) == "x"
assert simplify(("*", ("+", "x", "y"), ("-", 3, 3))) == 0
assert simplify(("-", ("+", "x", 1), ("+", "x", 1))) == 0
assert simplify(("/", ("*", "a", "b"), ("*", "a", "b"))) == 1
assert simplify(("-", 0, "x")) == ("neg", "x")
assert simplify(("neg", ("neg", ("+", "x", 0)))) == "x"
assert simplify(("neg", ("+", 2, 3))) == -5
assert simplify(("/", "x", ("-", 2, 2))) == ("/", "x", 0), "division by zero stays"
assert simplify(("+", ("*", 2, "x"), ("*", 3, 4))) == ("+", ("*", 2, "x"), 12)
assert simplify("x") == "x" and simplify(3.5) == 3.5

big = ("+", ("*", ("-", "x", 0), ("/", "y", "y")), ("-", 0, ("*", 0, "z")))
assert simplify(big) == "x"
assert evaluate(big, {"x": 4, "y": 2, "z": 9}) == evaluate(simplify(big), {"x": 4}) == 4

"✅ All OK!"

## Задача 6 – Отчет за продажби

Дадени са поръчки като `namedtuple`-и `Order(customer, product, quantity, price, month)`.

Напишете функциите **без нито един `for`/`while` цикъл** извън comprehensions – само с comprehensions, `map`, `filter`, `functools.reduce`, `sorted`, `zip`, `any`, `all`, `sum`, `max` и т.н.

* `revenue_by_customer(orders)` – речник `клиент → общ приход` (`quantity * price`), **подреден** по приход низходящо
* `top_products(orders, n)` – списък от `n` наредени двойки `(продукт, общо количество)` с най-голямо количество; при равенство – по име възходящо
* `loyal_customers(orders)` – сортиран списък от клиентите, които имат поръчка във **всеки** от месеците, които се срещат в данните
* `monthly_growth(orders)` – списък от тройки `(месец, приход, промяна)` по месеци възходящо, където `промяна` е процентната промяна спрямо предходния месец, закръглена до 1 знак (`None` за първия месец)
* `best_month_per_product(orders)` – речник `продукт → месецът с най-голямо количество продадени` (при равенство – по-ранният месец)

In [ ]:
from collections import namedtuple
from functools import reduce

Order = namedtuple("Order", ["customer", "product", "quantity", "price", "month"])


def revenue_by_customer(orders):
    pass  # your code here


def top_products(orders, n):
    pass  # your code here


def loyal_customers(orders):
    pass  # your code here


def monthly_growth(orders):
    pass  # your code here


def best_month_per_product(orders):
    pass  # your code here


# Tests:

orders = [
    Order("ana", "coffee", 2, 5.0, 1),
    Order("bob", "tea", 3, 3.0, 1),
    Order("ana", "cake", 1, 8.0, 2),
    Order("cvetan", "coffee", 4, 5.0, 2),
    Order("bob", "coffee", 1, 5.0, 2),
    Order("ana", "tea", 5, 3.0, 3),
    Order("bob", "cake", 2, 8.0, 3),
    Order("cvetan", "tea", 1, 3.0, 3),
]

assert revenue_by_customer(orders) == {"ana": 33.0, "bob": 30.0, "cvetan": 23.0}
assert list(revenue_by_customer(orders)) == ["ana", "bob", "cvetan"], "the dict should be ordered by revenue"
assert revenue_by_customer([]) == {}

assert top_products(orders, 2) == [("tea", 9), ("coffee", 7)]
assert top_products(orders, 10) == [("tea", 9), ("coffee", 7), ("cake", 3)]
assert top_products(orders + [Order("ana", "beer", 7, 2.0, 3)], 3) == [("tea", 9), ("beer", 7), ("coffee", 7)]

assert loyal_customers(orders) == ["ana", "bob"]
assert loyal_customers(orders[:2]) == ["ana", "bob"]
assert loyal_customers([]) == []

assert monthly_growth(orders) == [(1, 19.0, None), (2, 33.0, 73.7), (3, 34.0, 3.0)]
assert monthly_growth(orders[:1]) == [(1, 10.0, None)]

assert best_month_per_product(orders) == {"coffee": 2, "tea": 3, "cake": 3}
assert best_month_per_product(orders + [Order("bob", "cake", 1, 8.0, 2)]) == {"coffee": 2, "tea": 3, "cake": 2}, "earlier month wins ties"

"✅ All OK!"